process pool (managing queues, sentinels, and manual process spawning), this chapter is about the easy, elegant way Python provides out-of-the-box.

Too slow, no concurrncy

In [ ]:
import time
from pathlib import Path
from typing import Callable
import httpx

# Our "targets" - instead of country codes, we want Wikipedia articles to build an NLP corpus
WIKI_TOPICS = [
    'Natural_language_processing', 
    'Machine_learning', 
    'Deep_learning', 
    'Transformer_(machine_learning_model)', 
    'Large_language_model',
    'Reinforcement_learning',
    'Computer_vision'
]

# The Wikipedia REST API returns clean JSON summaries
BASE_URL = 'https://en.wikipedia.org/api/rest_v1/page/summary'
DEST_DIR = Path('nlp_corpus')

def save_text(text: str, filename: str) -> None:
    """Saves the scraped text to our local corpus directory."""
    (DEST_DIR / filename).write_text(text, encoding='utf-8')

def get_wiki_summary(topic: str) -> str:
    """Hits the API and extracts the plain text summary."""
    url = f'{BASE_URL}/{topic}'
    
    # httpx is modern and supports both sync (used here) and async
    resp = httpx.get(url, timeout=10.0, follow_redirects=True)
    resp.raise_for_status()  # Fails loudly if we get a 404 or 500 error
    
    # Parse the JSON and grab the 'extract' field which contains the plain text
    data = resp.json()
    return data.get('extract', '')

def download_many(topic_list: list[str]) -> int:
    """The sequential bottleneck: downloads one by one."""
    for topic in sorted(topic_list):
        text = get_wiki_summary(topic)
        save_text(text, f'{topic}.txt')
        
        # Print progress on the same line
        print(topic[:10] + '...', end=' ', flush=True)
        
    return len(topic_list)

def main(downloader: Callable[[list[str]], int]) -> None:
    DEST_DIR.mkdir(exist_ok=True) # Create the corpus folder
    t0 = time.perf_counter()
    
    count = downloader(WIKI_TOPICS)
    
    elapsed = time.perf_counter() - t0
    print(f'\n{count} articles scraped in {elapsed:.2f}s')

if __name__ == '__main__':
    main(download_many)

In [ ]:
import time
from concurrent import futures
from pathlib import Path
import httpx

# Our NLP corpus targets
WIKI_TOPICS = [
    'Natural_language_processing', 'Machine_learning', 'Deep_learning', 
    'Transformer_(machine_learning_model)', 'Large_language_model',
    'Reinforcement_learning', 'Computer_vision'
]

BASE_URL = 'https://en.wikipedia.org/api/rest_v1/page/summary'
DEST_DIR = Path('nlp_corpus')

def save_text(text: str, filename: str) -> None:
    (DEST_DIR / filename).write_text(text, encoding='utf-8')

def get_wiki_summary(topic: str) -> str:
    url = f'{BASE_URL}/{topic}'
    resp = httpx.get(url, timeout=10.0, follow_redirects=True)
    resp.raise_for_status()
    return resp.json().get('extract', '')

def download_one(topic: str) -> str:
    """The isolated worker function."""
    text = get_wiki_summary(topic)
    save_text(text, f'{topic}.txt')
    print(f"[{topic[:10]}... done]", end=' ', flush=True)
    return topic

def download_many(topic_list: list[str]) -> int:
    """The concurrent downloader using a ThreadPool."""
    # 1. Instantiate the executor
    with futures.ThreadPoolExecutor() as executor:
        # 2. Map the worker function to the list of inputs
        results = executor.map(download_one, sorted(topic_list))
        
    # 3. Consume the results iterator
    return len(list(results))

if __name__ == '__main__':
    DEST_DIR.mkdir(exist_ok=True)
    t0 = time.perf_counter()
    
    count = download_many(WIKI_TOPICS)
    
    elapsed = time.perf_counter() - t0
    print(f'\n{count} articles scraped concurrently in {elapsed:.2f}s')

The Context Manager (with statement): Opening the ThreadPoolExecutor in a with block ensures that the script will automatically call executor.shutdown(wait=True) when it's done. Your main program will pause right there and will not exit until every single worker thread has finished its job.

The Power of executor.map(): You don't have to manually create threads, build queues, or inject "poison pills." You just give map() your target function and your list of data. It automatically spins up the threads, feeds them the topics, and collects the returned strings.

Result Ordering: Even though the threads are firing simultaneously and finishing at different times (e.g., "Machine Learning" might return faster than "Deep Learning"), executor.map() is smart enough to reassemble the results iterator in the exact same order as the input list.

The GIL Bypass: When download_one calls httpx.get(), the underlying C implementation of the Python socket library explicitly drops the Global Interpreter Lock (GIL). Thread 1 goes to sleep waiting for Wikipedia's server, releasing the GIL so Thread 2 can immediately fire off its own HTTP request. All 7 network waits overlap.

In the previous example, executor.map() hid all the complexity. But in real-world data pipelines, you often need fine-grained control over individual tasks. To get that, we use executor.submit() and futures.as_completed().

In [ ]:
import time
from concurrent import futures
from pathlib import Path
import httpx

# We will use 5 topics, but only give the Executor 3 workers to force a bottleneck
WIKI_TOPICS = [
    'Algorithm', 'Backpropagation', 'Cybernetics', 'Data_mining', 'Expert_system'
]

BASE_URL = 'https://en.wikipedia.org/api/rest_v1/page/summary'
DEST_DIR = Path('nlp_corpus')

def get_wiki_summary(topic: str) -> str:
    url = f'{BASE_URL}/{topic}'
    resp = httpx.get(url, timeout=10.0, follow_redirects=True)
    resp.raise_for_status()
    return topic  # Just returning the topic name for this demonstration

def download_many(topic_list: list[str]) -> int:
    # 1. Start a pool with strictly 3 workers
    with futures.ThreadPoolExecutor(max_workers=3) as executor:
        
        to_do_list: list[futures.Future] = []
        
        # 2. Schedule the jobs (does NOT wait for them to finish)
        for topic in sorted(topic_list):
            # submit() gives us a "ticket" (Future) immediately
            future = executor.submit(get_wiki_summary, topic)
            to_do_list.append(future)
            print(f'Scheduled: {topic:15} -> {future}')
            
        print("\n--- All tasks scheduled. Waiting for results... ---\n")
        
        # 3. Process results EXACTLY as they finish, regardless of input order
        for count, future in enumerate(futures.as_completed(to_do_list), 1):
            res = future.result() # This extracts the actual return value
            print(f'Finished:  {res:15} -> {future}')
            
    return count

if __name__ == '__main__':
    download_many(WIKI_TOPICS)

What is a Future? (The "Claim Ticket"): When you drop your car off at a mechanic, they give you a ticket. You don't have your car, you have a promise that you will get your car later. That ticket is a Future. When you call executor.submit(), Python immediately hands you a Future object so the main for loop can keep moving without waiting for the Wikipedia network request.

Running vs. Pending: Because we gave the executor 5 topics but only 3 workers (max_workers=3), the system maxes out. If you run this code, the first 3 Futures will print as state=running. The last 2 will print as state=pending. They are sitting in the internal queue waiting for a thread to become available.

map() vs as_completed() (The Crucial Difference):

executor.map() returns results in the exact order you submitted them. If "Algorithm" takes 10 seconds to download, and "Backpropagation" takes 1 second, map() will pause and wait for 10 seconds, holding up the entire pipeline just to maintain the list order.

futures.as_completed() acts like a funnel. It yields whichever Future finishes first. If "Backpropagation" finishes in 1 second, it spits it out immediately so your program can save it to disk, even while "Algorithm" is still downloading.

In [ ]:
import time
from concurrent import futures

# Let's mock a corpus of documents with their character lengths.
# We sort them in descending order: massive documents first, tiny ones last.
CORPUS = [
    ("Massive_Wikipedia_Dump.txt", 50_000_000), # Will take a long time
    ("Medium_Article.txt", 5_000_000),
    ("Short_Blog.txt", 500_000),
    ("Tiny_Tweet.txt", 50_000)
]

def heavy_nlp_tokenization(doc_name: str, char_length: int) -> tuple[str, float]:
    """A mock function simulating heavy CPU string manipulation."""
    t0 = time.perf_counter()
    
    # Simulating a purely CPU-bound task (burning CPU cycles)
    _ = sum([x * x for x in range(char_length // 10)]) 
    
    elapsed = time.perf_counter() - t0
    return doc_name, elapsed

def process_corpus(corpus: list[tuple[str, int]]) -> None:
    # 1. Instantiate a Process pool instead of a Thread pool
    # If we don't specify max_workers, it defaults to our exact CPU core count.
    with futures.ProcessPoolExecutor() as executor:
        
        print("Starting heavy tokenization...")
        t0 = time.perf_counter()
        
        # 2. executor.map handles IPC (Inter-Process Communication) automatically
        # We submit the documents in descending order of size
        results = executor.map(heavy_nlp_tokenization, 
                               [doc for doc, _ in corpus], 
                               [size for _, size in corpus])
        
        # 3. Iterate through the results yielded by map()
        for doc_name, elapsed in results:
            print(f"Processed {doc_name:25} in {elapsed:.4f}s")
            
        print(f"\nTotal Pipeline Time: {time.perf_counter() - t0:.2f}s")

if __name__ == '__main__':
    process_corpus(CORPUS)

ProcessPool vs. ThreadPool: The API is identical. You literally just swap the word Thread for Process. But under the hood, Python is now bypassing the GIL by cloning the Python interpreter into entirely separate memory spaces. This is what you use when your ML pipeline is choked by data augmentations or regex parsing.

Massive Code Reduction: You no longer have to write JobQueue, ResultQueue, or Poison Pill logic. The Executor abstracts away all the Inter-Process Communication. The author notes this reduces boilerplate code by nearly 30%.

The executor.map Bottleneck (The "Hanging" Illusion): The text highlights a critical quirk about map(). It forces the output to match the exact order of the input list.

Because we passed the Massive_Wikipedia_Dump first, Worker 1 starts processing it.

Workers 2, 3, and 4 instantly finish the smaller documents.

Even though the small documents are done, map() will not yield them to your for loop. It blocks your main program, waiting silently for that massive document to finish.

Once the massive document finishes, your terminal will instantly dump the results for all the other documents at the exact same millisecond.

The Takeaway for Pipelines: If you are building a real-time data ingestion pipeline and you use .map() with a mix of huge and tiny payloads, your pipeline will look like it's freezing. If you want results to flow smoothly as soon as they are ready regardless of order, you must abandon map() and use the futures.as_completed() pattern we looked at in the previous section.

# To be continued